# Assignment: Preparing Data for Analysis (Modified Titanic)

![](https://github.com/kaopanboonyuen/2110446_DataScience_2021s2/raw/main/%20files/hw.png)

# 1) Load data & review the data

How many rows are there in the "titanic_to_student.csv"?

In [213]:
#Import the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [214]:
df = pd.read_csv('titanic_to_student.csv')

In [215]:
row_counts = df.shape[0]
print(f"There are {row_counts} rows")

There are 445 rows


In [216]:
# Set indeex
df = df.set_index('PassengerId')

In [217]:
# # Most use command
# df.shape
# df.info()
# df.isnull().sum()

# # At least must investigate our target variable (At least)
# # Other features are recommended to investigate
# df["num"].hist()
# # skewness, kurtosis --> -1,1
# df["col"].value_counts()

# # 1. EDA
# # 2. Narrow Down Features
# # 3. Data Prep (Impute missing, non_numeric -> numeric, remove outlier, transform, feat eng)
# # 4. Split Data
# train_test_split(X,Y,test_size=0.3, stratify=Purchase, random_state=12345) # Random train test แบบ fix ratio
# # แต่ทุกครั้งที่แบ่ง -> แบ่งแบบ Random ควรมส่ seed number เพื่อ control การสุ่ม (random_state)
# # data 20:80
# # train 20:80
# # test 20:80

# 2) Drop unqualified variables
2.1 Drop variables with missing > 50%

2.2 Check all columns except 'Age' and 'Fare' for flat values, drop the columns where flat value > 70%
From 2.1 and 2.2, how many columns do we have left?

Note: 
    
    -Ensure missing values are considered in your calculation. If you use normalize in .value_counts(), please include dropna=False.






In [218]:
# 2.1
print(df.isna().sum()*100/row_counts)
drop_threshold = len(df)*0.5 # 50%
df = df.dropna(thresh=drop_threshold, axis=1)
print(df.info())


Unnamed: 0     0.000000
Survived       2.921348
Pclass         6.966292
Name           2.696629
Sex            0.000000
Age           19.101124
SibSp          3.820225
Parch          0.000000
Ticket         4.269663
Fare           0.000000
Cabin         73.932584
Embarked      10.112360
dtype: float64
<class 'pandas.core.frame.DataFrame'>
Index: 445 entries, 2 to 890
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  445 non-null    int64  
 1   Survived    432 non-null    float64
 2   Pclass      414 non-null    float64
 3   Name        433 non-null    object 
 4   Sex         445 non-null    object 
 5   Age         360 non-null    float64
 6   SibSp       428 non-null    float64
 7   Parch       445 non-null    int64  
 8   Ticket      426 non-null    object 
 9   Fare        445 non-null    float64
 10  Embarked    400 non-null    object 
dtypes: float64(5), int64(2), object(4)
memory usage: 41.7+ KB
N

In [219]:
# 2.2
rem_col = []
for col in df.columns:
    top_count = df[col].value_counts().iloc[0]
    top_pct = top_count/df.shape[0]
    if top_pct > 0.7:
        top_val = df[col].value_counts().index[0]
        print(f"Remove {col}: {top_val} with {top_pct:.1%}")
        rem_col.append(col)

df = df.drop(rem_col, axis=1)

Remove Parch: 0 with 76.2%


In [220]:
print(f"We have {df.shape[1]} columns remaining")

We have 10 columns remaining


# 3) Remove all rows with missing target (the variable "Survived")

Remove all rows with missing targets (the variable "Survived")

How many rows do we have left?


In [221]:
before_drop = df["Survived"].isna().sum()
df = df.dropna(subset=['Survived'],axis=0)
print(f"Before: {before_drop}, After: {df['Survived'].isna().sum()}")

Before: 13, After: 0


# 4) Handle outliers 
Handle outliers

For the variable “Fare”, replace outlier values with the boundary values

If value < (Q1 - 1.5IQR), replace with (Q1 - 1.5IQR)

If value > (Q3 + 1.5IQR), replace with (Q3 + 1.5IQR)

What is the average (mean) of “Fare” after replacing the outliers (round 2 decimal points)?

Hint: Use function round(_, 2)



In [222]:
quartiles = df["Fare"].quantile([0.25,0.5,0.75])
[q1, q2, q3] = quartiles.values
iqr = q3 - q1
print(f"IQR: {iqr}")
print(f"Q1: {q1}, Q2:{q2}, Q3:{q3}")

print(f"Val > Upper Quartile: {sum(df["Fare"] > (q3 + 1.5 * iqr))} rows")
df.loc[df["Fare"] > (q3 + 1.5*iqr), "Fare"] = q3 + 1.5*iqr

print(f"Val < Lower Quartile: {sum(df["Fare"] < (q1 - 1.5 * iqr))} rows")
df.loc[df["Fare"] < (q1 - 1.5 * iqr), "Fare"] = q1 - 1.5 * iqr

print(f"Average of Fare is {df["Fare"].mean():.2f}")
# df["Fare"].hist()

IQR: 26.19165
Q1: 7.9177, Q2:15.2458, Q3:34.10935
Val > Upper Quartile: 55 rows
Val < Lower Quartile: 0 rows
Average of Fare is 26.23


# 5) Impute missing value

Impute missing value

For number type column, impute missing values with mean

What is the average (mean) of “Age” after imputing the missing values (round 2 decimal points)?

Hint: Use function round(_, 2)


In [223]:
number_cols = ["Age", "SibSp", "Fare"]
for col in number_cols:
    print(f"Column:{col} replace {df[col].isna().sum()} record with {df[col].mean()}")
    df.loc[df[col].isna(), col] = df[col].mean()

Column:Age replace 83 record with 29.332636103151863
Column:SibSp replace 17 record with 0.491566265060241
Column:Fare replace 0 record with 26.22834716435185


# 6) Convert categorical to numeric values

Convert categorical to numeric values

For the variable “Embarked”, perform the dummy coding.

What is the average (mean) of “Embarked_Q” after performing dummy coding (round 2 decimal points)?

Hint: Use function round(_, 2)



In [224]:

cat_cols = ["Sex", "Embarked"] # Pclass is already numeric, Ticket is behave like id
for col in cat_cols:
    if col == "Embarked":
        df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
    else:
        df[col] = df[col].astype('category').cat.codes
print("Columns After:", df.columns.tolist())
print(df.head())

Columns After: ['Unnamed: 0', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Ticket', 'Fare', 'Embarked_Q', 'Embarked_S']
             Unnamed: 0  Survived  Pclass  \
PassengerId                                 
2                     0       1.0     1.0   
4                     1       1.0     1.0   
6                     2       0.0     3.0   
8                     3       0.0     3.0   
10                    4       1.0     2.0   

                                                          Name  Sex  \
PassengerId                                                           
2            Cumings, Mrs. John Bradley (Florence Briggs Th...    0   
4                 Futrelle, Mrs. Jacques Heath (Lily May Peel)    0   
6                                             Moran, Mr. James    1   
8                               Palsson, Master. Gosta Leonard    1   
10                         Nasser, Mrs. Nicholas (Adele Achem)    0   

                   Age  SibSp    Ticket     Fare  Embarke

# 7) Partition data
Split train/test split with stratification using 70%:30% and random seed with 123

Show a proportion between survived (1) and died (0) in all data sets (total data, train, test)

What is the proportion of survivors (survived = 1) in the training data (round 2 decimal points)?

Hint: Use function round(_, 2), and train_test_split() from sklearn.model_selection




In [225]:
from sklearn.model_selection import train_test_split

In [236]:
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=123
)

total_props = sum(y==1)/sum(y==0)
train_props = sum(y_train == 1) / sum(y_train == 0)
test_props = sum(y_test == 1) / sum(y_test == 0)
print(f"Survived proportion total: {total_props}, train: {train_props}, test: {test_props}")
print(f"The proportions of surviors in the training data is {train_props:.2f}")

Survived proportion total: 0.7142857142857143, train: 0.6685082872928176, test: 0.8309859154929577
The proportions of surviors in the training data is 0.67
